In [1]:
!pip install transformers pillow pandas gradio torch torchvision

In [3]:
# 2. Clone Repositori Git Anda
!git clone https://github.com/Brian071/Text-to-Image-Retrieval.git

# 3. MASUK KE FOLDER REPOSITORI
%cd Text-to-Image-Retrieval

# 4. Membuat direktori dataset
!mkdir -p dataset

# 5. Mengunduh dataset dari AWS S3 langsung ke dalam folder 'dataset'
print("Mengunduh dataset... (Ini mungkin memakan waktu beberapa menit)")
!wget -q https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-listings.tar -O dataset/abo-listings.tar
!wget -q https://amazon-berkeley-objects.s3.amazonaws.com/archives/abo-images-small.tar -O dataset/abo-images-small.tar

# 6. Mengekstrak isi dataset
print("Mengekstrak dataset...")
!tar -xf dataset/abo-listings.tar -C dataset
!tar -xf dataset/abo-images-small.tar -C dataset

# 7. Bersihkan file .tar untuk menghemat RAM/Disk Colab
!rm dataset/abo-listings.tar
!rm dataset/abo-images-small.tar

# 8. Hapus paket yang berpotensi konflik
!pip uninstall -y transformers torch torchvision numpy scipy pandas seaborn

# 9. Install dependensi secara aman (termasuk Gradio untuk Web UI)
print("Menginstal dependensi...")
!pip install "numpy<2.0.0" scipy pandas seaborn fastapi uvicorn torch torchvision transformers[torch] qdrant-client>=1.10.0 Pillow "opencv-python<4.11.0" optuna segment-anything python-multipart urllib3>=2 matplotlib gradio

# Install Localtunnel (opsional jika Anda butuh port forwarding khusus, Gradio share=True biasanya sudah cukup)
!npm install -g localtunnel

print("✅ PERSIAPAN SELESAI! Silakan jalankan Cell 2.")

fatal: destination path 'Text-to-Image-Retrieval' already exists and is not an empty directory.
/content/Text-to-Image-Retrieval
Mengunduh dataset... (Ini mungkin memakan waktu beberapa menit)
Mengekstrak dataset...
Found existing installation: transformers 5.13.0
Uninstalling transformers-5.13.0:
  Successfully uninstalled transformers-5.13.0
Found existing installation: torch 2.12.1
Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1
Found existing installation: torchvision 0.27.1
Uninstalling torchvision-0.27.1:
  Successfully uninstalled torchvision-0.27.1
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: scipy 1.17.1
Uninstalling scipy-1.17.1:
  Successfully uninstalled scipy-1.17.1
Found existing installation: pandas 3.0.3
Uninstalling pandas-3.0.3:
  Successfully uninstalled pandas-3.0.3
Found existing installation: seaborn 0.13.2
Uninstalling seaborn-0.13.2:
  Successfully uni

In [4]:
# Membuat folder tempat metadata seharusnya berada
!mkdir -p dataset/images/metadata

# Mengunduh file images.csv.gz yang terlupakan
!wget https://amazon-berkeley-objects.s3.amazonaws.com/images/metadata/images.csv.gz -P dataset/images/metadata/

--2026-07-04 09:20:40--  https://amazon-berkeley-objects.s3.amazonaws.com/images/metadata/images.csv.gz
Resolving amazon-berkeley-objects.s3.amazonaws.com (amazon-berkeley-objects.s3.amazonaws.com)... 16.15.253.155, 54.231.192.97, 16.15.255.107, ...
Connecting to amazon-berkeley-objects.s3.amazonaws.com (amazon-berkeley-objects.s3.amazonaws.com)|16.15.253.155|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6430535 (6.1M) [text/csv]
Saving to: ‘dataset/images/metadata/images.csv.gz.1’

images.csv.gz.1     100%[===================>]   6.13M  28.6MB/s    in 0.2s    

2026-07-04 09:20:40 (28.6 MB/s) - ‘dataset/images/metadata/images.csv.gz.1’ saved [6430535/6430535]



In [8]:
import torch
import pandas as pd
import os
import gc
import gradio as gr
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

# 1. KONFIGURASI
DATA_DIR = "dataset"
METADATA_PATH = os.path.join(DATA_DIR, "images", "metadata", "images.csv.gz")
MODEL_NAME = "openai/clip-vit-base-patch32"

MAX_SAMPLES = 9000
BATCH_SIZE = 128

def get_real_path(img_path):
    p1 = os.path.join(DATA_DIR, "images", "small", str(img_path))
    p2 = os.path.join(DATA_DIR, str(img_path))
    if os.path.exists(p2): return p2
    if os.path.exists(p1): return p1
    return p2

# 2. CLASS MODEL HANDLER (Versi Bersih Tanpa Hack Dimensi)
class CLIPHandler:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Memuat model AI di {self.device}...")
        self.model = CLIPModel.from_pretrained(MODEL_NAME).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(MODEL_NAME)

    def get_image_embeddings(self, image_paths):
        all_embeddings = []

        print(f"Memulai ekstraksi {len(image_paths)} gambar dengan sistem Batching...")
        for i in range(0, len(image_paths), BATCH_SIZE):
            batch_paths = image_paths[i:i+BATCH_SIZE]
            images = [Image.open(p).convert("RGB") for p in batch_paths]

            inputs = self.processor(images=images, return_tensors="pt").to(self.device)

            with torch.no_grad():
                # get_image_features menjamin output matriks berukuran (batch, 512)
                embeds = self.model.get_image_features(**inputs)

                # Normalisasi
                embeds = embeds / embeds.norm(dim=-1, keepdim=True)
                all_embeddings.append(embeds.cpu())

            del images, inputs, embeds
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()

            if (i + BATCH_SIZE) % (BATCH_SIZE * 5) == 0 or (i + BATCH_SIZE) >= len(image_paths):
                print(f"Progress: {min(i + BATCH_SIZE, len(image_paths))} / {len(image_paths)} selesai diproses.")

        # Menggabungkan kembali matriks
        return torch.cat(all_embeddings, dim=0).to(self.device)

    def get_text_embedding(self, text):
        inputs = self.processor(text=[text], return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            # get_text_features menjamin output matriks (batch, 512)
            embeds = self.model.get_text_features(**inputs)
            embeds = embeds / embeds.norm(dim=-1, keepdim=True)
        return embeds

# 3. CLASS RETRIEVER
class ImageRetriever:
    def __init__(self):
        self.clip = CLIPHandler()
        print("Memuat dataset...")
        try:
            self.df = pd.read_csv(METADATA_PATH).head(MAX_SAMPLES)
            self.df['absolute_path'] = self.df['path'].apply(get_real_path)
            self.df = self.df[self.df['absolute_path'].apply(os.path.exists)]
            self.image_paths = self.df['absolute_path'].tolist()

            print(f"✅ Berhasil menemukan {len(self.image_paths)} gambar fisik!")

        except Exception as e:
            print(f"❌ Error memuat data: {e}")
            self.image_paths = []
            self.df = pd.DataFrame()

        if self.image_paths:
            self.image_embeddings = self.clip.get_image_embeddings(self.image_paths)
            print("✅ Ekstraksi selesai! Sistem siap digunakan.")
        else:
            self.image_embeddings = None

    def search(self, query, top_k=4):
        if self.image_embeddings is None or len(self.image_paths) == 0:
            return []

        text_emb = self.clip.get_text_embedding(query)

        # --- SOLUSI ANTI-ERROR DIMENSI ---
        # Mengganti cosine_similarity dengan Perkalian Matriks (Dot Product)
        # text_emb @ image_embeddings.T otomatis mencocokkan matriks dengan sempurna
        similarities = (text_emb @ self.image_embeddings.T).squeeze()

        # Jaga-jaga jika hasilnya hanya ada 1 gambar (skalar), ubah kembali menjadi array
        if similarities.dim() == 0:
            similarities = similarities.unsqueeze(0)

        top_indices = similarities.argsort(descending=True)[:top_k].cpu().numpy()
        return [self.image_paths[i] for i in top_indices]

# 4. GRADIO WEB APP
print("\n=== Memulai Inisialisasi Sistem ===")
retriever = ImageRetriever()

def search_images_ui(query):
    result_paths = retriever.search(query, top_k=4)
    results = []

    for p in result_paths:
        if os.path.exists(p):
            row = retriever.df[retriever.df['absolute_path'] == p]

            if 'item_name' in row.columns:
                teks = str(row['item_name'].values[0])
            else:
                teks = f"ID Path: {str(row['path'].values[0])}"

            results.append((p, teks))

    return results

with gr.Blocks() as demo:
    gr.Markdown("<h1 style='text-align: center;'>🛍️ Visual Search (Fix Matriks)</h1>")

    with gr.Row():
        with gr.Column(scale=3):
            query_input = gr.Textbox(label="Cari Produk", placeholder="Ketik apa saja (misal: shoes, black bag)")
        with gr.Column(scale=1):
            search_btn = gr.Button("🔍 Cari", variant="primary")

    gallery = gr.Gallery(label="Hasil Pencarian", show_label=True, columns=2, rows=2, object_fit="contain", height=600)

    search_btn.click(fn=search_images_ui, inputs=query_input, outputs=gallery)
    query_input.submit(fn=search_images_ui, inputs=query_input, outputs=gallery)

print("\n=== Menjalankan Web App ===")
demo.launch(share=True, debug=True, theme=gr.themes.Soft())


=== Memulai Inisialisasi Sistem ===
Memuat model AI di cpu...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Memuat dataset...
✅ Berhasil menemukan 9000 gambar fisik!
Memulai ekstraksi 9000 gambar dengan sistem Batching...


AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'norm'